# Resumo do projeto — C318 no-show

Notebook-resumo da disciplina **C318 – Fundamentos de Machine Learning** (Inatel).
Lê exclusivamente artefatos congelados de `reports/` e, opcionalmente, confirma os
números de limpeza a partir do CSV bruto.
**Nenhuma célula treina ou avalia modelo.**

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

# resolve o módulo src a partir da raiz do repositório
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

BASE = ROOT / "reports" / "modelo-final"
REPORTS = ROOT / "reports"

## 1. Contexto, objetivo e enquadramento

**Problema de negócio.** Faltas sem aviso em consultas do SUS desperdiçam vagas e
prolongam filas. O objetivo é prever, *no momento do agendamento*, a probabilidade de
falta de cada paciente, para direcionar confirmações ativas ou reagendamentos aos casos
de maior risco.

**Objetivo de ciência de dados.** Classificação binária supervisionada: `no_show = 1`
(faltou) ou `0` (compareceu). Métricas principais: *Average Precision* (AP / PR-AUC) e
recall nos 20% de maior risco (política de contato K = 20%).

**Enquadramento.** Uma única cidade (Vitória/ES, 2016), ~6 semanas de consultas, ~20%
de faltas. Resultado esperado: *score* de risco por agendamento (não rótulo fixo), para
que a unidade escolha quantos pacientes contatar conforme sua capacidade operacional.

## 2. Dados

In [ ]:
# confirma os números de limpeza se o CSV bruto estiver disponível;
# caso contrário exibe os valores documentados no README (seção 6)
RAW_PATH = ROOT / "data" / "raw" / "KaggleV2-May-2016.csv"

if RAW_PATH.exists():
    from src.data import load_raw, clean
    from src.features import build_features

    df = build_features(clean(load_raw(str(RAW_PATH))))

    # apenas agregados — nenhuma linha individual é exibida
    print(f"\nAgendamentos após limpeza: {len(df):>7,}")
    print(f"Pacientes distintos:       {df['patient_id'].nunique():>7,}")
    taxa = df["no_show"].mean()
    print(f"Taxa de falta:             {taxa:>8.2%}  ({df['no_show'].sum():,} de {len(df):,})")
    dmin = df["appointment_day"].dt.normalize().min().date()
    dmax = df["appointment_day"].dt.normalize().max().date()
    print(f"Período das consultas:     {dmin} → {dmax}")
else:
    print("CSV bruto não encontrado em data/raw/KaggleV2-May-2016.csv")
    print("Para baixar:")
    print("  kaggle datasets download -d joniarroba/noshowappointments -p data/raw/ --unzip")
    print()
    print("Números documentados (README, seção 6; src/data.py):")
    print("  CSV bruto:              110.527 agendamentos")
    print("  Após limpeza:           109.889  (−638: 6 inválidos + 632 duplicatas)")
    print("  Pacientes distintos:     62.298")
    print("  Taxa de falta:           20,10%  (22.091 faltas)")
    print("  Período das consultas:   29/04/2016 → 08/06/2016")

## 3. Protocolo de avaliação

- **Partição por paciente** (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`):
  87.720 de desenvolvimento e 22.169 de teste reservado (4.497 faltas, prevalência 20,29%).
  Nenhum paciente aparece nos dois conjuntos.
- **Teste reservado usado uma única vez**, após congelamento da escolha do modelo em
  `reports/modelo-final/selecao_congelada.json` (20/nov/2026).
  O teste *não* entra em EDA, CV nem escolha de hiperparâmetros.
- **Validação cruzada:** `StratifiedGroupKFold(n_splits=5)` sobre o desenvolvimento,
  sem pacientes compartilhados entre folds.
- **Histórico do paciente:** para cada agendamento, conta consultas do mesmo paciente com
  `appointment_day` *estritamente anterior* ao `scheduled_day` — evita vazamento de dados.
  Só 28% dos agendamentos têm histórico disponível (janela curta de ~6 semanas).
- **`sms_received` fora do modelo:** enviado após o agendamento (posterior ao momento de
  previsão). Entra apenas nas análises exploratórias.

## 4. Comparação em validação cruzada

Cinco candidatos avaliados em 5 folds sobre os 87.720 agendamentos de desenvolvimento,
com e sem as 4 variáveis de histórico do paciente.

In [ ]:
# lê a tabela de validação cruzada (cv_candidatos.csv) e exibe com e sem histórico
cv = pd.read_csv(BASE / "cv_candidatos.csv")

COLS = ["candidate", "ap_mean", "ap_std", "roc_auc_mean",
        "precision_at_20_mean", "recall_at_20_mean"]
LABELS = ["Candidato", "AP médio", "± dp", "ROC-AUC", "Prec@20%", "Rec@20%"]
FMT = {"AP médio": "{:.4f}", "± dp": "{:.4f}", "ROC-AUC": "{:.4f}",
       "Prec@20%": "{:.1%}", "Rec@20%": "{:.1%}"}

for hist, titulo in [(False, "Sem histórico"), (True, "Com histórico")]:
    sub = cv[cv["use_history"] == hist][COLS].copy()
    sub.columns = LABELS
    sub = sub.sort_values("AP médio", ascending=False).reset_index(drop=True)
    print(f"\n{titulo}")
    display(sub.style.format(FMT))

In [ ]:
# ablação do histórico: diferença de AP ao incluir as 4 variáveis de histórico
abl = pd.read_csv(BASE / "cv_ablacao_historico.csv")
abl = abl.sort_values("delta_ap", ascending=False).reset_index(drop=True)
abl.columns = ["Candidato", "AP sem histórico", "AP com histórico", "ΔAP"]
display(abl.style.format({
    "AP sem histórico": "{:.4f}",
    "AP com histórico": "{:.4f}",
    "ΔAP": "{:+.4f}",
}))

## 5. Escolha do modelo

XGBoost com histórico alcançou o maior AP médio e menor desvio-padrão entre folds.
A escolha foi **congelada antes do teste** em `selecao_congelada.json`.
O limiar principal isola os 20% de maior risco nas predições *out-of-fold* do
desenvolvimento.

In [ ]:
# exibe a escolha congelada: modelo, parâmetros, limiar e data de congelamento
frozen = json.loads((BASE / "selecao_congelada.json").read_text(encoding="utf-8"))

print(f"Modelo:              {frozen['model']}")
print(f"Histórico:           {frozen['use_history']}")
print(f"Parâmetros:          {frozen['params']}")
print(f"Limiar top-20%:      {frozen['limiar_top20']:.4f}  ← política de contato principal")
print(f"Limiar F1 (alt.):    {frozen['limiar_f1']:.4f}")
print(f"AP OOF (dev):        {frozen['oof_ap']:.4f}  ({frozen['oof_n_rows']:,} linhas)")
print(f"Data congelamento:   {frozen['data_congelamento']}")

## 6. Resultado único no teste

Todos os candidatos retreinados no desenvolvimento completo (87.720 linhas, com
histórico) e avaliados **uma única vez** no teste reservado (22.169 agendamentos,
4.497 faltas).

In [ ]:
# tabela de resultados no teste com ganho sobre o acaso
teste_csv = pd.read_csv(BASE / "teste_final.csv")
teste_json = json.loads((BASE / "teste_final.json").read_text(encoding="utf-8"))

# recall@20% do dummy é a referência de acaso (20% aleatório)
dummy_rec = next(r for r in teste_json if r["candidate"] == "dummy")["recall_at_20"]
tab = teste_csv[["candidate", "ap", "roc_auc", "precision_at_20", "recall_at_20"]].copy()
tab["ganho_acaso"] = tab["recall_at_20"] / dummy_rec
tab = tab.sort_values("ap", ascending=False).reset_index(drop=True)
tab.columns = ["Candidato", "AP", "ROC-AUC", "Prec@20%", "Rec@20%", "Ganho/acaso"]

display(tab.style.format({
    "AP": "{:.4f}", "ROC-AUC": "{:.4f}",
    "Prec@20%": "{:.1%}", "Rec@20%": "{:.1%}",
    "Ganho/acaso": "{:.2f}×",
}))

In [ ]:
# detalhe do XGBoost: matriz de confusão com o limiar congelado
xgb_res = next(r for r in teste_json if r["candidate"] == "xgboost")
cm = xgb_res["confusion_matrix"]
n_total      = sum(x for row in cm for x in row)
n_faltas     = cm[1][0] + cm[1][1]
n_sinalizados = cm[0][1] + cm[1][1]   # FP + TP
n_capturadas  = cm[1][1]               # TP

limiar = frozen["limiar_principal"]
print(f"Limiar:                    {limiar:.4f}  (top-20% do desenvolvimento)")
print(f"Agendamentos sinalizados:  {n_sinalizados:>6,}  ({n_sinalizados/n_total:.1%} do teste)")
print(f"Faltas capturadas (TP):    {n_capturadas:>6,}  de {n_faltas:,}"
      f"  → recall {n_capturadas/n_faltas:.1%}")
print(f"Precisão nos contatos:     {xgb_res['precision_at_20']:.1%}")
print(f"Ganho sobre o acaso:       {xgb_res['recall_at_20']/dummy_rec:.2f}×")

df_cm = pd.DataFrame(
    cm,
    index=["Real: compareceu", "Real: faltou"],
    columns=["Prev: compareceu", "Prev: faltou"],
)
print()
display(df_cm)

## 7. Interpretação das variáveis

### SHAP — importância por variável original

Para variáveis com OHE, os valores SHAP de todas as colunas da mesma variável são
somados *com sinal* em cada linha da amostra; depois calcula-se a média do valor
absoluto. Isso evita inflação de variáveis com muitas categorias (ex.: `neighbourhood`).
Valores em log-odds.

In [ ]:
# tabela de importância pelo SHAP (mean|SHAP| agregado por variável original)
shap_df = pd.read_csv(BASE / "shap_por_variavel.csv")
shap_df.columns = ["Rank", "Variável", "mean|SHAP| (log-odds)"]
display(shap_df.style.format({"mean|SHAP| (log-odds)": "{:.4f}"}))

In [ ]:
# figura de barras do SHAP por variável original
fig_path = BASE / "figures" / "shap_por_variavel.png"
if fig_path.exists():
    img = mpimg.imread(str(fig_path))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(f"Figura não encontrada: {fig_path}")

### Sensibilidade ao bairro

`neighbourhood` lidera o SHAP, mas tem 81 categorias one-hot. Esta análise verifica
quanto o modelo perde sem ela e com agrupamento dos bairros de baixo volume (< 100
agendamentos no fold).

In [ ]:
# sensibilidade ao bairro em 5 folds do modelo final (exploratório — modelo não muda)
sens = pd.read_csv(BASE / "sensibilidade_bairro.csv")
sens.columns = ["Variante", "AP médio (5 folds)", "± dp"]
display(sens.style.format({"AP médio (5 folds)": "{:.4f}", "± dp": "{:.4f}"}))

# delta calculado a partir dos dados — sem digitar valores à mão
ap_ref = sens.loc[sens["Variante"] == "modelo_final", "AP médio (5 folds)"].values[0]
ap_sem = sens.loc[sens["Variante"] == "sem_neighbourhood", "AP médio (5 folds)"].values[0]
ap_agr = sens.loc[sens["Variante"] == "bairros_agrupados", "AP médio (5 folds)"].values[0]
print(f"\nRemover neighbourhood:   ΔAP {ap_sem - ap_ref:+.4f}")
print(f"Agrupar bairros raros:   ΔAP {ap_agr - ap_ref:+.4f}  (dentro do dp de ~{sens.iloc[0]['± dp']:.4f})")

### Análise de erros

Taxa de FN alta em grupos de baixa prevalência é **consequência esperada** da
política de contato seletivo (top 20%), não falha do modelo: contatos se concentram
onde o risco é maior. A coluna *taxa de falta real* permite distinguir os dois casos.

In [ ]:
# taxas de FP e FN por dimensão e grupo; apenas agregados por grupo, sem linhas individuais
err = pd.read_csv(BASE / "error_analysis.csv")

for dim in err["dimension"].unique():
    sub = err[err["dimension"] == dim][
        ["group", "falta_rate", "fp_rate", "fn_rate", "n_total"]
    ].copy()
    sub.columns = ["Grupo", "Taxa falta real", "FP rate", "FN rate", "N"]
    print(f"\n--- {dim} ---")
    display(sub.reset_index(drop=True).style.format(
        {"Taxa falta real": "{:.1%}", "FP rate": "{:.1%}", "FN rate": "{:.1%}"}
    ))

## 8. Impacto de negócio

Tradução das métricas do teste em vagas recuperadas e custo de contato
(seção 7 do README). **Todos os valores são estimativas condicionadas às hipóteses
da tabela.** O efeito do contato (*e*) não foi medido neste dataset.

In [ ]:
# hipóteses e comparação de modelos por 1.000 agendamentos, política top-20%, e=20%
imp = pd.read_csv(REPORTS / "impacto_negocio.csv")

# parâmetros comuns a todas as linhas
p = imp.iloc[0]
print("Hipóteses:")
print(f"  Valor da vaga:         US$ {p['valor_vaga_usd']:.0f}  (fonte secundária, histórica)")
print(f"  Custo por contato:     R$ {p['custo_por_contato_brl']:.2f}  (hipótese de trabalho)")
print(f"  Contatos por 1.000:    {p['contatos_por_1000']:.0f}  (política K = 20%)")
print()

# comparação e=20%, câmbio R$5, corte top20
top20_e20 = imp[(imp["corte"] == "top20") & (imp["e"] == 0.2)].copy()
cols = ["modelo", "capturadas_por_1000", "vagas_recuperadas_por_1000",
        "vagas_recuperadas_aleatorio_por_1000",
        "custo_liquido_brl_cambio_5_0", "break_even_e_cambio_5_0"]
top20_e20 = top20_e20[cols].rename(columns={
    "modelo": "Modelo",
    "capturadas_por_1000": "Faltas cap./1000",
    "vagas_recuperadas_por_1000": "Vagas rec./1000 (e=20%)",
    "vagas_recuperadas_aleatorio_por_1000": "Acaso (e=20%)",
    "custo_liquido_brl_cambio_5_0": "Custo líquido R$/1000 (câmbio 5)",
    "break_even_e_cambio_5_0": "Ponto equil. e",
})
display(top20_e20.reset_index(drop=True).style.format({
    "Faltas cap./1000": "{:.1f}",
    "Vagas rec./1000 (e=20%)": "{:.1f}",
    "Acaso (e=20%)": "{:.1f}",
    "Custo líquido R$/1000 (câmbio 5)": "{:,.0f}",
    "Ponto equil. e": "{:.1%}",
}))

In [ ]:
# sensibilidade ao efeito do contato: XGBoost, top-20%, câmbio R$5
xgb_imp = imp[(imp["modelo"] == "xgboost") & (imp["corte"] == "top20")][
    ["e", "vagas_recuperadas_por_1000", "vagas_recuperadas_aleatorio_por_1000",
     "custo_liquido_brl_cambio_5_0"]
].rename(columns={
    "e": "e (efeito do contato)",
    "vagas_recuperadas_por_1000": "Vagas rec./1000 (modelo)",
    "vagas_recuperadas_aleatorio_por_1000": "Vagas rec./1000 (acaso)",
    "custo_liquido_brl_cambio_5_0": "Custo líquido R$/1000",
})
display(xgb_imp.reset_index(drop=True).style.format({
    "e (efeito do contato)": "{:.0%}",
    "Vagas rec./1000 (modelo)": "{:.1f}",
    "Vagas rec./1000 (acaso)": "{:.1f}",
    "Custo líquido R$/1000": "{:,.0f}",
}))

## 9. Limitações e próximos passos

**Dados**
- Uma única cidade (Vitória/ES) e o ano de 2016, em janela curta (~6 semanas de
  consultas). Resultados não representam o SUS como um todo.
- Histórico disponível em ~28% dos agendamentos (janela curta). Em produção seria
  necessário registro contínuo por paciente.
- 14 grupos de duplicatas com alvo divergente: fonte potencial de ruído.

**Modelo e avaliação**
- Uma única partição de teste e sem intervalo de confiança. A diferença XGBoost × RF
  (~0,2 p.p. de recall@20%) não é conclusiva.
- Scores com `scale_pos_weight` não são probabilidades calibradas; o limiar não deve
  ser lido como probabilidade real de falta.
- `neighbourhood` domina o SHAP por fragmentação one-hot; para comunicação externa,
  enfatizar `waiting_days` e `age` como variáveis mais generalizáveis.
- Modelo final não avaliado na validação temporal.

**Impacto de negócio**
- Efeito do contato (*e*), valor da vaga e custo por contato são **hipóteses**.
  Nenhum foi medido neste dataset.
- O cálculo assume que a vaga liberada é preenchida.

**Próximos passos**
1. Medir *e* com piloto controlado.
2. Levantar custo real de contato e valor da vaga na unidade.
3. Validar em dados de outras unidades ou períodos.
4. Calibrar os scores do XGBoost.
5. Testar `same_day` como variável e avaliar limiar de contato mais amplo.